# OrderPulse — Spark-Based E-Commerce Order Analytics

**Week 6 Assignment | Celebal Excellence Internship (CEI) 2026**
**Track:** Data Engineering — Apache Spark Fundamentals
**Author:** Nikhil Rohilla

---

## 1. Project Introduction

Most of the Spark exercises I'd seen before this assignment revolved around the same worn-out retail
`orders.csv` with three columns and a `groupBy("category").sum("amount")` at the end. Instead of
recycling that pattern, this notebook is built around a small but realistic **order-processing pipeline**
for a fictional online marketplace called **OrderPulse**.

The idea is to walk through the pipeline the way an actual data engineer would approach it on day one of
a new project: load the raw feed, understand what's actually inside it, clean up the mess, and only then
start answering business questions. Every Spark concept required by the Week 6 assignment sheet — schema
inference, filtering, casting, derived columns, Parquet round-trips, `explain()`, and so on — is
demonstrated as a natural step in that workflow rather than as an isolated textbook exercise.

By the end of this notebook you'll have:

- A working `SparkSession` reading a raw CSV order feed
- A cleaned, correctly-typed DataFrame ready for analytics
- Answers to the 15 theory/practical questions from the assignment sheet, worked into the relevant section
- Parquet and CSV outputs written back to disk for downstream consumption


## 2. Business Problem

OrderPulse operates across five regional fulfilment zones (**North, South, East, West, Central**) and
routes every order through one of five warehouses. The operations team currently exports a daily CSV dump
from the order-management system, and nobody downstream trusts it completely — customer IDs go missing on
guest checkouts, payment mode is sometimes left blank when a transaction is retried, and price fields
arrive as raw strings rather than numeric types.

Before any dashboard or reconciliation job can consume this feed, we need a Spark job that:

1. Loads the raw feed and understands its shape and quality issues
2. Produces a cleaned, strongly-typed version of the dataset
3. Answers recurring operational questions — high-value completed orders, regional payment preferences,
   revenue after tax, orders stuck without a customer reference — directly from the DataFrame
4. Persists the cleaned dataset in a columnar format (Parquet) so later stages (BI tools, ML feature
   pipelines) don't have to repeat this cleanup

This notebook *is* that job, built incrementally.


## 3. Dataset Description

The source file `ecommerce_orders.csv` contains **60 order records** exported from OrderPulse's order
system, covering six product categories and five regional zones. Columns:

| Column | Description |
|---|---|
| `Order_ID` | Unique order identifier (e.g. `ORD1001`) |
| `Customer_ID` | Customer reference — can be blank for guest/unlinked checkouts |
| `Product_Name` | Name of the purchased item |
| `Category` | Product category (Electronics, Fashion, Books, etc.) |
| `Brand` | Brand/manufacturer label |
| `Region` | Fulfilment zone the order was shipped from |
| `Order_Status` | Completed / Pending / Cancelled / Returned / Shipped |
| `Order_Date` | Date the order was placed |
| `Quantity` | Units ordered |
| `Unit_Price` | Price per unit before discount |
| `Discount` | Discount percentage applied |
| `Total_Amount` | Final payable amount after discount |
| `Payment_Mode` | Payment channel used — can be blank on retried transactions |
| `Warehouse` | Warehouse code that fulfilled the order |

The dataset deliberately includes a handful of missing `Customer_ID` and `Payment_Mode` values so the
cleaning section later in the notebook has something real to fix.


## 4. Environment Setup

Nothing exotic here — just the PySpark SQL functions we'll lean on throughout the notebook. Keeping the
import list explicit (rather than `from pyspark.sql.functions import *`) makes it obvious later exactly
which transformation each line is using.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)
from pyspark.sql.functions import col, round as spark_round, when, count as spark_count, avg, sum as spark_sum

print("PySpark imports ready.")

PySpark imports ready.


## 5. Spark Session Initialization

**Q1 (Assignment). Driver, Cluster Manager, and Executor — what does each actually do?**

It helps to think of a Spark job the way you'd think of a construction project. The **Driver** is the site
engineer: it holds the blueprint (your code), decides how the work should be broken up, and keeps track of
progress. It never lays a single brick itself.

The **Cluster Manager** (YARN, Kubernetes, Mesos, or Spark's own Standalone manager) is the site's resource
office — the Driver asks it "I need 4 workers with 2 cores each," and the Cluster Manager finds and
allocates that capacity from whatever machines are available.

The **Executors** are the actual bricklayers. They run on the worker nodes, execute the tasks the Driver
assigns them, hold data partitions in memory or on local disk for the duration of the job, and report
results and status back to the Driver.

```
        Driver
          │   (plans stages, builds DAG, tracks task status)
          ▼
  Cluster Manager
          │   (allocates executor containers on worker nodes)
          ▼
      Executors
   (run tasks, cache partitions, shuffle data, return results)
```

When we call `.master("local[*]")` below, we're telling Spark to skip the external Cluster Manager
entirely and run the Driver and all Executors as threads inside this one JVM — perfect for a laptop-scale
notebook, but architecturally it's still the same three roles.

**Q13 (Assignment). Client Mode vs Cluster Mode**

The distinction is really just: *where does the Driver process live?*

- **Client Mode** — the Driver runs on the machine you launched `spark-submit` from (your laptop, an
  edge node, this notebook's kernel). Convenient for interactive work because logs and `print()` output
  land right in front of you, but if that machine disconnects, the job dies with it.
- **Cluster Mode** — the Driver itself is handed off to run *inside* the cluster, on one of the worker
  nodes, managed by the Cluster Manager. This is the standard choice for production jobs submitted via a
  scheduler, since the job survives independently of the machine that submitted it.

This notebook runs in client mode by construction — the Jupyter kernel *is* the Driver process.


In [2]:
spark = (
    SparkSession.builder
    .appName("OrderPulse-EcommerceAnalytics")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Application ID:", spark.sparkContext.applicationId)

26/07/26 19:22:52 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/07/26 19:22:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/26 19:22:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.1
Application ID: local-1785093777093


## 6. Data Loading

**Q3 (Assignment). Reading a CSV with a header row and inferred schema**

The line below is the direct answer to Q3 — `header=True` tells Spark the first row is column names
rather than data, and `inferSchema=True` makes Spark do a pass over the file to guess appropriate types
(integers, doubles, dates) instead of defaulting every column to `string`.


In [3]:
orders_inferred = spark.read.csv(
    "ecommerce_orders.csv",
    header=True,
    inferSchema=True
)

orders_inferred.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit_Price: double (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Total_Amount: double (nullable = true)
 |-- Payment_Mode: string (nullable = true)
 |-- Warehouse: string (nullable = true)



Inferred schemas are convenient, but notice that `Unit_Price` came through as a clean `double` — that
only happened because our sample data is well-formed. Real order-management exports are rarely that kind;
prices often arrive as strings (sometimes with currency symbols or formatting quirks upstream). To make the
casting exercise in Section 10 meaningful rather than a no-op, we load the working copy of the dataset
again with an **explicit schema** that keeps `Unit_Price` as a string, mirroring what you'd actually get
from a less disciplined source system.


In [4]:
order_schema = StructType([
    StructField("Order_ID", StringType(), True),
    StructField("Customer_ID", StringType(), True),
    StructField("Product_Name", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Brand", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Order_Status", StringType(), True),
    StructField("Order_Date", DateType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Unit_Price", StringType(), True),   # kept as String on purpose — see note above
    StructField("Discount", IntegerType(), True),
    StructField("Total_Amount", DoubleType(), True),
    StructField("Payment_Mode", StringType(), True),
    StructField("Warehouse", StringType(), True),
])

orders_df = spark.read.csv(
    "ecommerce_orders.csv",
    header=True,
    schema=order_schema
)

orders_df.printSchema()

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit_Price: string (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Total_Amount: double (nullable = true)
 |-- Payment_Mode: string (nullable = true)
 |-- Warehouse: string (nullable = true)



## 7. Data Understanding

**Q15 (Assignment). Why `.show(5)` instead of `.collect()` on a huge dataset?**

`.collect()` pulls every single row from every executor back to the Driver's memory as a local Python list.
On a multi-terabyte dataset that's an instant way to crash the Driver — it was never sized to hold the
whole dataset, only to coordinate the job. `.show(n)` instead asks Spark to compute just enough partitions
to produce `n` rows and print them directly on the executors' side of the boundary, so the Driver only ever
receives a small, bounded amount of formatted text. It's the difference between asking someone to *mail you
their entire filing cabinet* versus asking them to *read you the top five folders over the phone*.

With that in mind, all exploration below uses `.show()` and `.count()` rather than `.collect()`, even
though our 60-row dataset would technically survive either.


In [5]:
record_count = orders_df.count()
print(f"Total order records: {record_count}")

orders_df.show(5, truncate=False)

Total order records: 60


+--------+-----------+-------------------------+----------------+----------------+-------+------------+----------+--------+----------+--------+------------+------------+---------+
|Order_ID|Customer_ID|Product_Name             |Category        |Brand           |Region |Order_Status|Order_Date|Quantity|Unit_Price|Discount|Total_Amount|Payment_Mode|Warehouse|
+--------+-----------+-------------------------+----------------+----------------+-------+------------+----------+--------+----------+--------+------------+------------+---------+
|ORD1001 |CUST137    |Yoga Mat 6mm             |Sports & Fitness|FitCore         |East   |Completed   |2026-02-05|6       |985.72    |15      |5027.17     |Debit Card  |WH-DEL-01|
|ORD1002 |CUST126    |Data Engineering Handbook|Books           |PageTurner Press|North  |Completed   |2026-05-10|5       |217.58    |0       |1087.9      |Wallet      |WH-KOL-05|
|ORD1003 |CUST148    |Robotic Vacuum Cleaner   |Home & Kitchen  |ComfortLiving   |North  |Returned  

In [6]:
orders_df.select("Category").distinct().show()
orders_df.groupBy("Order_Status").count().orderBy(col("count").desc()).show()

+--------------------+
|            Category|
+--------------------+
|               Books|
|         Electronics|
|    Sports & Fitness|
|      Home & Kitchen|
|             Fashion|
|Beauty & Personal...|
+--------------------+



+------------+-----+
|Order_Status|count|
+------------+-----+
|   Completed|   30|
|     Shipped|   11|
|     Pending|    9|
|    Returned|    5|
|   Cancelled|    5|
+------------+-----+



## 8. Data Cleaning

Before trusting any aggregation on this feed, we should know exactly how dirty it is. The snippet below
counts, per column, how many values are either SQL `NULL` or an empty string (CSV exports frequently blur
that distinction — a truly missing value and `""` are not the same thing in Spark's eyes, so we check for
both).


In [7]:
null_audit = orders_df.select([
    spark_count(when(col(c).isNull() | (col(c) == ""), c)).alias(c)
    for c in orders_df.columns
])

null_audit.show()

+--------+-----------+------------+--------+-----+------+------------+----------+--------+----------+--------+------------+------------+---------+
|Order_ID|Customer_ID|Product_Name|Category|Brand|Region|Order_Status|Order_Date|Quantity|Unit_Price|Discount|Total_Amount|Payment_Mode|Warehouse|
+--------+-----------+------------+--------+-----+------+------------+----------+--------+----------+--------+------------+------------+---------+
|       0|          3|           0|       0|    0|     0|           0|         0|       0|         0|       0|           0|           2|        0|
+--------+-----------+------------+--------+-----+------+------------+----------+--------+----------+--------+------------+------------+---------+



As expected from the source system's known quirks, `Customer_ID` and `Payment_Mode` are the only columns
with gaps. Dropping those rows would throw away otherwise-valid revenue data, so instead we fill them with
explicit sentinel values — `"UNKNOWN"` for orders with no linked customer (typical of guest checkouts) and
`"Not Specified"` where the payment channel wasn't captured. This keeps every row usable for aggregation
while making the gap visible rather than silently coercing it to `null`.


In [8]:
orders_clean = orders_df.na.fill({
    "Customer_ID": "UNKNOWN",
    "Payment_Mode": "Not Specified"
})

orders_clean.filter(
    (col("Customer_ID") == "UNKNOWN") | (col("Payment_Mode") == "Not Specified")
).show(6, truncate=False)

+--------+-----------+-------------------------+--------------+--------------------+-------+------------+----------+--------+----------+--------+------------+-------------+---------+
|Order_ID|Customer_ID|Product_Name             |Category      |Brand               |Region |Order_Status|Order_Date|Quantity|Unit_Price|Discount|Total_Amount|Payment_Mode |Warehouse|
+--------+-----------+-------------------------+--------------+--------------------+-------+------------+----------+--------+----------+--------+------------+-------------+---------+
|ORD1008 |UNKNOWN    |Non-Stick Cookware Set   |Home & Kitchen|DomusCraft          |West   |Completed   |2026-06-13|6       |5233.07   |15      |26688.66    |Net Banking  |WH-DEL-01|
|ORD1016 |CUST135    |Memory Foam Pillow       |Home & Kitchen|HomeEase            |North  |Returned    |2026-01-18|5       |7008.59   |0       |35042.95    |Not Specified|WH-HYD-04|
|ORD1024 |UNKNOWN    |Atomic Habits            |Books         |PageTurner Press    |W

## 9. Schema Inspection

A quick sanity check before moving into transformations — confirming the cleaned DataFrame's structure and
column-level types didn't drift during the fill operation above (they shouldn't; `na.fill` never changes a
column's declared type, only its null values).


In [9]:
orders_clean.printSchema()
print("\nColumn list:", orders_clean.columns)
print("Row count after cleaning:", orders_clean.count())

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = false)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Unit_Price: string (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Total_Amount: double (nullable = true)
 |-- Payment_Mode: string (nullable = false)
 |-- Warehouse: string (nullable = true)


Column list: ['Order_ID', 'Customer_ID', 'Product_Name', 'Category', 'Brand', 'Region', 'Order_Status', 'Order_Date', 'Quantity', 'Unit_Price', 'Discount', 'Total_Amount', 'Payment_Mode', 'Warehouse']


Row count after cleaning: 60


## 10. DataFrame Operations

**Q11 (Assignment). Transformations vs Actions**

This distinction is the single most important mental model in Spark. A **transformation**
(`.select()`, `.filter()`, `.withColumn()`, `.orderBy()`, ...) never touches data immediately — it just
adds a new node to a logical plan and hands you back a new DataFrame describing *what should happen*.
Nothing actually executes. An **action** (`.show()`, `.count()`, `.collect()`, `.write()`, ...) is what
finally tells Spark "okay, now go compute this," triggering the whole chain of transformations built up
so far.

```
Transformations
      ↓
Lazy Evaluation   (Spark just records the plan, does no work yet)
      ↓
DAG Creation      (plan is compiled into a Directed Acyclic Graph of stages)
      ↓
Action            (.show(), .count(), .write() ... triggers execution)
      ↓
Execution         (DAG is scheduled onto executors as real tasks)
```

Two transformations used below: `.select()` and `.withColumn()`. Two actions used below: `.show()` and
`.count()`.

**Q2 (Assignment). How does Lazy Evaluation improve performance on chained operations?**

Because nothing runs until an action fires, Spark gets to see the *entire* chain of transformations at
once before deciding how to execute any of it. That means it can optimize globally rather than
step-by-step: pushing filters down close to the data source, combining several `.select()`/`.filter()`
calls into a single scan, skipping columns you never asked for, and avoiding materializing intermediate
DataFrames that would otherwise sit around in memory doing nothing. If Spark executed eagerly (line by
line, like a normal Python script), it would have no opportunity to see the bigger picture and would waste
time and memory building full intermediate results after every single `.filter()` or `.select()` call.


In [10]:
# Q5 — select specific columns for a specific category
electronics_view = orders_clean.select("Order_ID", "Product_Name", "Total_Amount") \
    .where(col("Category") == "Electronics")

electronics_view.show(5, truncate=False)

+--------+------------------------+------------+
|Order_ID|Product_Name            |Total_Amount|
+--------+------------------------+------------+
|ORD1005 |Portable SSD 1TB        |105577.92   |
|ORD1006 |Wireless Earbuds Pro    |8998.24     |
|ORD1009 |27-inch Monitor         |42912.42    |
|ORD1021 |Smart Fitness Band      |10760.69    |
|ORD1023 |Noise Cancelling Headset|19398.97    |
+--------+------------------------+------------+
only showing top 5 rows



**Q8 (Assignment). Filtering with AND** — completed orders above a revenue threshold:

In [11]:
high_value_completed = orders_clean.filter(
    (col("Order_Status") == "Completed") & (col("Total_Amount") > 10000)
)

print("High-value completed orders:", high_value_completed.count())
high_value_completed.select("Order_ID", "Region", "Total_Amount").show(5)

High-value completed orders: 7


+--------+-------+------------+
|Order_ID| Region|Total_Amount|
+--------+-------+------------+
| ORD1005|Central|   105577.92|
| ORD1008|   West|    26688.66|
| ORD1009|   East|    42912.42|
| ORD1023|  North|    19398.97|
| ORD1026|Central|    21401.49|
+--------+-------+------------+
only showing top 5 rows



**Q14 (Assignment). Filtering with OR** — orders that either ship from the North zone or were flagged
high priority in some way; here we use payment channel as the second condition since our schema doesn't
carry an explicit priority flag:

In [12]:
north_or_upi = orders_clean.filter(
    (col("Region") == "North") | (col("Payment_Mode") == "UPI")
)

print("Matching rows:", north_or_upi.count())
north_or_upi.select("Order_ID", "Region", "Payment_Mode").show(5)

Matching rows: 16


+--------+------+-------------+
|Order_ID|Region| Payment_Mode|
+--------+------+-------------+
| ORD1002| North|       Wallet|
| ORD1003| North|          UPI|
| ORD1007|  East|          UPI|
| ORD1014|  East|          UPI|
| ORD1016| North|Not Specified|
+--------+------+-------------+
only showing top 5 rows



**Q6 (Assignment). Renaming a column and casting its type**

This is exactly the scenario Section 6 set up: `Unit_Price` currently lives as a `string` because of how we
defined the read schema. Renaming it to something clearer and casting it to `double` in one chained call:


In [13]:
orders_revised = orders_clean.withColumnRenamed("Unit_Price", "unit_price_raw") \
    .withColumn("unit_price", col("unit_price_raw").cast(DoubleType())) \
    .drop("unit_price_raw")

orders_revised.printSchema()
orders_revised.select("Order_ID", "unit_price").show(5)

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = false)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Total_Amount: double (nullable = true)
 |-- Payment_Mode: string (nullable = false)
 |-- Warehouse: string (nullable = true)
 |-- unit_price: double (nullable = true)



+--------+----------+
|Order_ID|unit_price|
+--------+----------+
| ORD1001|    985.72|
| ORD1002|    217.58|
| ORD1003|   4091.22|
| ORD1004|   1542.25|
| ORD1005|  22226.93|
+--------+----------+
only showing top 5 rows



**Q10 (Assignment). Derived column** — adding an 18% tax on top of the unit price to get the actual
payable `final_price`:

In [14]:
orders_priced = orders_revised.withColumn(
    "final_price",
    spark_round(col("unit_price") * 1.18, 2)
)

orders_priced.select("Order_ID", "unit_price", "final_price").show(5)

+--------+----------+-----------+
|Order_ID|unit_price|final_price|
+--------+----------+-----------+
| ORD1001|    985.72|    1163.15|
| ORD1002|    217.58|     256.74|
| ORD1003|   4091.22|    4827.64|
| ORD1004|   1542.25|    1819.85|
| ORD1005|  22226.93|   26227.78|
+--------+----------+-----------+
only showing top 5 rows



## 11. Business Queries

With a cleaned, correctly-typed DataFrame in hand, we can finally answer the kind of questions the
operations team actually asks. `.distinct()` and `.orderBy()` — both required by the assignment sheet —
show up naturally here.


In [15]:
# Which warehouses are actually in play?
orders_priced.select("Warehouse").distinct().orderBy("Warehouse").show()

+---------+
|Warehouse|
+---------+
|WH-BLR-02|
|WH-DEL-01|
|WH-HYD-04|
|WH-KOL-05|
|WH-MUM-03|
+---------+



In [16]:
# Top 5 highest-value orders overall
orders_priced.orderBy(col("Total_Amount").desc()) \
    .select("Order_ID", "Category", "Region", "Total_Amount") \
    .show(5)

+--------+--------------+-------+------------+
|Order_ID|      Category| Region|Total_Amount|
+--------+--------------+-------+------------+
| ORD1005|   Electronics|Central|   105577.92|
| ORD1009|   Electronics|   East|    42912.42|
| ORD1016|Home & Kitchen|  North|    35042.95|
| ORD1017|Home & Kitchen|  South|    32450.64|
| ORD1059|Home & Kitchen|   East|    31704.19|
+--------+--------------+-------+------------+
only showing top 5 rows



In [17]:
# Average order value and total revenue per region
orders_priced.groupBy("Region") \
    .agg(
        spark_round(avg("Total_Amount"), 2).alias("avg_order_value"),
        spark_round(spark_sum("Total_Amount"), 2).alias("total_revenue")
    ) \
    .orderBy(col("total_revenue").desc()) \
    .show()

+-------+---------------+-------------+
| Region|avg_order_value|total_revenue|
+-------+---------------+-------------+
|Central|        18771.9|     187719.0|
|   East|       10369.31|    155539.64|
|  South|        9172.31|    137584.64|
|  North|       10698.11|    106981.06|
|   West|        5299.26|     52992.57|
+-------+---------------+-------------+



In [18]:
# Preferred payment mode per category (row count as a simple proxy for preference)
orders_priced.groupBy("Category", "Payment_Mode") \
    .count() \
    .orderBy("Category", col("count").desc()) \
    .show(15)

+--------------------+----------------+-----+
|            Category|    Payment_Mode|count|
+--------------------+----------------+-----+
|Beauty & Personal...|          Wallet|    2|
|Beauty & Personal...|Cash on Delivery|    2|
|Beauty & Personal...|     Net Banking|    2|
|Beauty & Personal...|             UPI|    1|
|Beauty & Personal...|     Credit Card|    1|
|Beauty & Personal...|      Debit Card|    1|
|               Books|     Net Banking|    4|
|               Books|Cash on Delivery|    3|
|               Books|          Wallet|    3|
|               Books|     Credit Card|    2|
|               Books|             UPI|    1|
|               Books|      Debit Card|    1|
|         Electronics|             UPI|    2|
|         Electronics|      Debit Card|    2|
|         Electronics|          Wallet|    2|
+--------------------+----------------+-----+
only showing top 15 rows



## 12. File Format Conversion

**Q4 (Assignment). CSV vs Parquet — row-based vs columnar, and why it matters**

CSV is a **row-based** format — every field of a row sits next to each other on disk, so to read even one
column, Spark still has to stream past every other column, for every row. Parquet is **columnar** — all
values of a single column are stored contiguously, block by block. If a downstream query only needs
`Total_Amount` and `Region` out of fourteen columns, Parquet lets Spark skip reading the other twelve
entirely. Parquet also stores per-column statistics (min/max, null counts) and applies compression far more
effectively, since similar values (a `Region` column with five repeating strings) sit next to each other.
For anything read more than once — which is the entire point of persisting a cleaned dataset — Parquet
wins on both I/O volume and file size.

**Q9 (Assignment). Predicate Pushdown in Parquet**

Because Parquet stores column-level min/max statistics per row-group, Spark can look at a filter like
`Total_Amount > 10000` *before* reading the data block and simply skip any row-group whose max value is
already below 10000 — the block is never even decompressed into memory. This is "pushing the predicate
down" to the storage layer instead of applying it in Spark after everything is loaded. For a filter that
eliminates most of the dataset, predicate pushdown can turn a full-table scan into reading a small fraction
of the actual file.

Writing the cleaned DataFrame to Parquet:


In [19]:
orders_priced.write.mode("overwrite").parquet("parquet_data")
print("Parquet write complete → parquet_data/")

Parquet write complete → parquet_data/


Reading it back to confirm the round-trip preserved both data and schema:

In [20]:
orders_from_parquet = spark.read.parquet("parquet_data")
orders_from_parquet.printSchema()
orders_from_parquet.show(3)

root
 |-- Order_ID: string (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Brand: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Date: date (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: integer (nullable = true)
 |-- Total_Amount: double (nullable = true)
 |-- Payment_Mode: string (nullable = true)
 |-- Warehouse: string (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- final_price: double (nullable = true)



+--------+-----------+--------------------+----------------+----------------+------+------------+----------+--------+--------+------------+------------+---------+----------+-----------+
|Order_ID|Customer_ID|        Product_Name|        Category|           Brand|Region|Order_Status|Order_Date|Quantity|Discount|Total_Amount|Payment_Mode|Warehouse|unit_price|final_price|
+--------+-----------+--------------------+----------------+----------------+------+------------+----------+--------+--------+------------+------------+---------+----------+-----------+
| ORD1001|    CUST137|        Yoga Mat 6mm|Sports & Fitness|         FitCore|  East|   Completed|2026-02-05|       6|      15|     5027.17|  Debit Card|WH-DEL-01|    985.72|    1163.15|
| ORD1002|    CUST126|Data Engineering ...|           Books|PageTurner Press| North|   Completed|2026-05-10|       5|       0|      1087.9|      Wallet|WH-KOL-05|    217.58|     256.74|
| ORD1003|    CUST148|Robotic Vacuum Cl...|  Home & Kitchen|   Comfort

**Q12 (Assignment). Parquet → filter nulls → write CSV**

A common downstream pattern: read the Parquet layer, drop any row missing a customer reference (rather
than the `"UNKNOWN"` sentinel — here we simulate the raw ask literally by filtering true nulls on
`Customer_ID`), and hand off a plain CSV extract to a team that only consumes flat files.


In [21]:
orders_for_export = orders_from_parquet.filter(col("Customer_ID").isNotNull())

orders_for_export.write.mode("overwrite").option("header", True).csv("processed_csv")
print("Filtered CSV export complete → processed_csv/")
print("Rows exported:", orders_for_export.count())

Filtered CSV export complete → processed_csv/


Rows exported: 60


## 13. Performance Discussion

**Q7 (Assignment). How does the Lineage Graph (DAG) provide fault tolerance?**

Spark never stores a copy of every intermediate result the way a traditional database might. Instead, each
RDD/DataFrame remembers its **lineage** — the exact sequence of transformations that produced it from the
original data source. If an executor holding a partition dies mid-job, Spark doesn't need a backup copy of
that partition; it simply looks up the lineage graph, finds which transformations built that partition, and
recomputes just that slice of data on a different executor. This is why Spark can tolerate worker failures
without checkpointing everything to disk by default — the DAG itself *is* the recovery mechanism.

Below, `.explain()` prints the physical plan Spark actually built for our high-value-completed-orders
filter from Section 10. Notice the `PushedFilters` line — this is predicate pushdown from Q9 in action,
visible directly in the plan rather than just described in theory.


In [22]:
high_value_completed.explain()

== Physical Plan ==
*(1) Project [Order_ID#45, coalesce(Customer_ID#46, UNKNOWN) AS Customer_ID#384, Product_Name#47, Category#48, Brand#49, Region#50, Order_Status#51, Order_Date#52, Quantity#53, Unit_Price#54, Discount#55, Total_Amount#56, coalesce(Payment_Mode#57, Not Specified) AS Payment_Mode#385, Warehouse#58]
+- *(1) Filter (((isnotnull(Order_Status#51) AND isnotnull(Total_Amount#56)) AND (Order_Status#51 = Completed)) AND (Total_Amount#56 > 10000.0))
   +- FileScan csv [Order_ID#45,Customer_ID#46,Product_Name#47,Category#48,Brand#49,Region#50,Order_Status#51,Order_Date#52,Quantity#53,Unit_Price#54,Discount#55,Total_Amount#56,Payment_Mode#57,Warehouse#58] Batched: false, DataFilters: [isnotnull(Order_Status#51), isnotnull(Total_Amount#56), (Order_Status#51 = Completed), (Total_Am..., Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/claude/Week-6/ecommerce_orders.csv], PartitionFilters: [], PushedFilters: [IsNotNull(Order_Status), IsNotNull(Total_Amount), EqualTo(Orde

**Performance notes from this run:**

- `spark.sql.shuffle.partitions` was lowered from Spark's default of 200 down to 4 at session start. The
  default is tuned for cluster-scale shuffles across hundreds of gigabytes; on a 60-row local dataset, 200
  tiny partitions would mean far more scheduling overhead than actual work.
- All filters in this notebook (`Order_Status`, `Total_Amount`, `Region`, `Payment_Mode`) were applied
  directly on Parquet reads where possible, letting predicate pushdown do the heavy lifting instead of
  loading full DataFrames into memory first.
- `.show()` was used throughout instead of `.collect()` for exactly the reason discussed in Section 7 (Q15)
  — no need to materialize results on the Driver when we only ever wanted a preview.


## 14. Conclusion

Starting from a 60-row raw export with realistic messiness — blank customer references, blank payment
modes, and a price column intentionally kept as a string — this notebook built a complete, if small-scale,
Spark ETL pass: schema-aware loading, null auditing and remediation, type correction, tax-inclusive
derived pricing, a handful of genuinely useful business aggregations, and a Parquet-backed persistence
layer with a downstream CSV export.

Every one of the 15 questions from the Week 6 assignment sheet was answered in the section where it was
contextually relevant rather than as a disconnected Q&A block — the goal being that this notebook reads
as a working analytics job first, and an assignment submission second.

**Next steps if this were a real production pipeline:** partition the Parquet output by `Region` or
`Order_Date` for faster downstream reads, add schema validation on ingest rather than assuming the CSV
always matches `order_schema`, and replace the local `SparkSession` with a cluster-mode submission once
data volume actually justifies it.


In [23]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
